# NH PLUG 클라이언트 퀵스타트

`cluefin_openapi.nhplug` 클라이언트를 실제로 호출해 보는 노트북입니다.

## 사전 준비

- 리포지토리 루트의 `.env.test` 에 `NHPLUG_APP_KEY` / `NHPLUG_SECRET_KEY` / `NHPLUG_ENV` 가 있어야 합니다.
- 실행: 리포지토리 루트에서 `uv run --with jupyter jupyter lab` 후 이 노트북을 엽니다.

## ⚠️ 반드시 알아야 할 것

- **토큰 발급은 운영 도메인 전용**이고 서버에서 1회/초로 제한되며, 불필요한 재발급은 계좌에 보안 알림을 발생시킵니다. 반드시 `Auth.generate()` (파일 캐시 사용)만 사용하세요 — 만료 전까지 실제 발급은 1회입니다.
- `NHPLUG_ENV=dev` 는 모의투자(moapi), `prod` 는 **운영(실계좌 — 주문이 실제 체결됨)** 입니다. 토큰 하나로 양쪽 도메인 모두 호출됩니다.
- 해외주식 **시세**(gbstock quote) 4종은 운영 도메인 전용입니다 — 모의투자(moapi)는 `IGW40019` 로 거부합니다.
- 모의투자 성공 응답코드는 `00000` 이 아니라 `XA102`("모의투자 조회가 완료되었습니다") 일 수 있습니다. `SUCCESS_RSP_CODES` 로 판정하세요.

## 1. 인증 — 토큰 발급 (파일 캐시 우선)

In [ ]:
import os

import dotenv
from pydantic import SecretStr

from cluefin_openapi.nhplug._auth import Auth

# 현재 디렉터리에서 위로 올라가며 리포지토리 루트의 .env.test 를 찾는다
dotenv.load_dotenv(dotenv.find_dotenv(".env.test", usecwd=True))

app_key = os.environ["NHPLUG_APP_KEY"]
secret_key = os.environ["NHPLUG_SECRET_KEY"]
env = os.getenv("NHPLUG_ENV", "dev")  # dev = 모의투자(moapi), prod = 운영(실계좌!)

auth = Auth(app_key=app_key, secret_key=SecretStr(secret_key))
token = auth.generate()  # 캐시된 토큰이 있으면 재발급하지 않는다
print("token expires_in:", token.expires_in)

## 2. HttpClient 생성과 응답 출력 헬퍼

In [ ]:
from pprint import pprint

from cluefin_openapi.nhplug._http_client import HttpClient
from cluefin_openapi.nhplug._model import SUCCESS_RSP_CODES

client = HttpClient(
    token=token.access_token,
    app_key=auth.app_key,
    secret_key=auth.secret_key,
    env=env,
)
print("env:", client.env, "→", client.base_url)


def show(response, keys=None):
    """응답의 성공 여부와 바디를 출력한다. keys 로 output 필드를 골라볼 수 있다."""
    body = response.body
    ok = body.rsp_cd in SUCCESS_RSP_CODES
    print(f"[{'OK' if ok else 'FAIL'}] rsp_cd={body.rsp_cd} rsp_msg={body.rsp_msg}")
    dumped = body.model_dump(exclude={"rsp_cd", "rsp_msg"}, exclude_none=True)
    if keys:
        dumped = {k: dumped.get(k) for k in keys}
    pprint(dumped, sort_dicts=False)
    return response

## 3. 계좌 목록 조회

모의투자(dev)는 `acct_type == "03"` 계좌만, 운영(prod)은 `01`/`02` 계좌만 유효합니다.

In [ ]:
accounts = client.common.get_account_list().body.output_0 or []
for a in accounts:
    print(a.acct_type, a.acct_no)

wanted = ("03",) if client.env == "dev" else ("01", "02")
account = next(a.acct_no for a in accounts if a.acct_type in wanted)
print("사용할 계좌:", account)

## 4. 국내주식 시세 (계좌번호 불필요)

In [ ]:
# 주식현재가 시세 — 삼성전자
res = show(client.krstock_quote.current_price(market_cd="KRX", iem_cd="005930"))
print("현재가:", res.body.output_0.stck_prpr)

In [ ]:
from datetime import date

# 기간별시세 — 최근 30개 일봉
res = client.krstock_quote.period(
    market_cd="KRX",
    iem_cd="005930",
    gubun="1",  # 일봉
    edate=date.today().strftime("%Y%m%d"),
    array_cnt="30",
)
print("rsp_cd:", res.body.rsp_cd)
for row in (res.body.output_1 or [])[:5]:
    print(row.model_dump(exclude_none=True))

In [ ]:
# 주식현재가 체결 · 일자별
res = client.krstock_quote.current_execution(market_cd="KRX", iem_cd="005930")
print("rsp_cd:", res.body.rsp_cd)
for row in (res.body.output_0 or [])[:5]:  # 최근 체결 틱
    print(row.model_dump(exclude_none=True))

res = client.krstock_quote.current_daily(market_cd="KRX", iem_cd="005930")
print("rsp_cd:", res.body.rsp_cd)
for row in (res.body.output_0 or [])[:5]:  # 일자별 주가
    print(row.model_dump(exclude_none=True))

In [ ]:
# 주식현재가 투자자 — 일자별 기관/외국인/개인 매매 동향 (최근 10건)
res = client.krstock_quote.current_investor(market_cd="KRX", iem_cd="005930", array_cnt="10")
print("rsp_cd:", res.body.rsp_cd)
for row in (res.body.output_0 or [])[:5]:
    print(row.model_dump(exclude_none=True))

In [ ]:
# 시간외현재가 · 시간외일자별주가
res = client.krstock_quote.after_hours_current(iem_cd="005930")
print("rsp_cd:", res.body.rsp_cd, "| output_0:", res.body.output_0 and res.body.output_0.model_dump(exclude_none=True))

res = client.krstock_quote.current_after_hours_daily(
    iem_cd="005930",
    date=date.today().strftime("%Y%m%d"),
    array_cnt="10",
    maxavg="5",
    gubun="1",
)
print("rsp_cd:", res.body.rsp_cd)
for row in (res.body.output_1 or [])[:3]:
    print(row.model_dump(exclude_none=True))

In [ ]:
# ETF/ETN 현재가 · ETF 구성종목시세 — KODEX 200 (069500)
res = client.krstock_quote.etf_current(iem_cd="069500")
print("rsp_cd:", res.body.rsp_cd, "| NAV:", res.body.output_3 and res.body.output_3.model_dump(exclude_none=True))

res = client.krstock_quote.etf_components(iem_cd="069500")
print("rsp_cd:", res.body.rsp_cd)
for row in (res.body.output_0 or [])[:5]:
    print(row.model_dump(exclude_none=True))

## 5. 국내주식 계좌 조회

In [ ]:
# 주식잔고조회
show(
    client.krstock_inquiry.balance(
        act_no=account,
        bnc_bse_cd="1",  # 체결기준
        ltg_aot_dit_cd="9",  # 전체
        aet_bse="1",  # 순자산
        qut_dit_cd="UNT",  # 통합시세
    )
);

In [ ]:
# 투자계좌자산현황 — 예수금·평가금액 확인은 여기서
show(
    client.krstock_inquiry.asset_status(
        act_no=account,
        eal_aly_cd="2",  # 시가평가
        aet_bse="1",  # 순자산
        qut_dit_cd="UNT",
    )
);

In [ ]:
# 매수가능수량 — 시장가 기준이라 주문가격 입력이 필요 없다
show(
    client.krstock_inquiry.buyable_quantity(
        act_no=account,
        iem_cd="005930",
        ost_dit_cd="1",  # 현금
        nmn_pr_tp_cd="05",  # 시장가
    )
);

## 6. 해외주식 계좌 조회 (모의투자 지원)

In [ ]:
# 해외주식 잔고
show(
    client.overseas_stock_inquiry.balance(
        act_no=account,
        qut_iqr_dit_cd="9",  # 전체
        fc_sec_trd_nat_cd="200",  # 미국
        cur_cd="KRW",
    )
);

In [ ]:
# 해외주식 매수가능금액 — AAPL, 원화 기준
show(
    client.overseas_stock_inquiry.buyable_amount(
        act_no=account,
        pcs_dit="1",  # 매수가능금액조회
        fc_sec_trd_nat_cd="200",  # 미국
        iem_cd="AAPL",
        wtm_cur_knd_cd="2",  # 원화
        oss_orr_knd_cd="1",  # GTS(미국시장주문)
        ahi_nmn_pr_tp_cd="03",  # 시장가
    )
);

## 7. 해외주식 시세 — ⚠️ 운영 도메인 전용

`/gbstock/quote/v1/*` 4종은 모의투자(moapi)에서 제공되지 않고 `IGW40019` 를 반환합니다.
아래 셀은 `NHPLUG_ENV=prod` 일 때만 실행됩니다. (시세 조회는 read-only 라 실계좌에도 안전합니다.)

In [ ]:
is_prod = client.env == "prod"
if not is_prod:
    print("모의투자(dev)에서는 gbstock 시세가 제공되지 않습니다 — NHPLUG_ENV=prod 로 실행하세요.")

if is_prod:
    # 해외주식 현재가상세 — AAPL (종목명은 iem_nm 으로 내려온다)
    res = show(client.overseas_stock_quote.current(iem_cd="AAPL"))

    # 해외주식 체결추이 — 일별 최근 10건
    res = client.overseas_stock_quote.execution_trend(
        period_type="2",  # 2.일별 (1.틱)
        req_cnt=10,
        iem_cd="AAPL",
    )
    print("rsp_cd:", res.body.rsp_cd)
    for row in (res.body.output_0 or [])[:5]:
        print(row.model_dump(exclude_none=True))

In [ ]:
if is_prod:
    # 해외주식 기간별시세(개별종목) — AAPL 최근 30개 일봉
    res = client.overseas_stock_quote.period(
        iem_cd="AAPL",
        end_dt=date.today().strftime("%Y%m%d"),
        count="0030",
        maxavg="005",
        gubun="3",  # 3.일 (4.주 5.월)
        xtick="0001",
        today_cls="1",  # 당일조회
        market_cls="1",  # 정규장
    )
    print("rsp_cd:", res.body.rsp_cd)
    for row in (res.body.output_0 or [])[:5]:
        print(row.model_dump(exclude_none=True))

    # 해외주식 기간별시세(지수·환율) — S&P 500 최근 30개 일봉
    res = client.overseas_stock_quote.symbol_index_fx_period(
        iem_cd="SPX",
        end_dt=date.today().strftime("%Y%m%d"),
        array_cnt="0030",
        maxavg="005",
        gubun="1",  # 1.일 (2.주 3.월)
        today_cls="0",  # 전체조회
    )
    print("rsp_cd:", res.body.rsp_cd)
    for row in (res.body.output_1 or [])[:5]:
        print(row.model_dump(exclude_none=True))
else:
    print("모의투자(dev)에서는 gbstock 시세가 제공되지 않습니다 — NHPLUG_ENV=prod 로 실행하세요.")

## 8. 국내주식 주문 — 🚨 직접 확인 후 주석 해제

**`NHPLUG_ENV=prod` 면 실제 주문이 체결됩니다.** 아래 예시는 모의투자(dev)에서만 실행되도록
가드했고, 그래도 기본은 주석 처리해 둡니다. 모의투자 주문은 장 운영시간(09:00–15:30 KST)에만 접수됩니다.

In [ ]:
# assert client.env == "dev", "운영 계좌에서는 이 셀을 실행하지 마세요"
#
# # 삼성전자 1주 시장가 매수 (모의투자)
# res = show(
#     client.krstock_order.cash_buy(
#         act_no=account,
#         iem_cd="005930",
#         orr_qty=1,
#         nmn_pr_tp_cd="05",  # 시장가
#         rmt_mkt_cd="KRX",
#         sor_mkt_sli_yn="N",
#     )
# )
#
# # 당일 주문체결 확인
# from datetime import date
# show(
#     client.krstock_inquiry.daily_order_execution(
#         act_no=account,
#         orr_dt=date.today().strftime("%Y%m%d"),
#         ost_cns_dit="0",  # 전체
#     )
# )